# Final Results Summary

This notebook consolidates the final experimental results of the Turkish Legal RAG project.

It summarizes:
1. Retrieval and RAG pipeline improvements.
2. Evaluation drift analysis.
3. Starlar LLM fine-tuning v2 results.
4. Final selected system configuration.
5. Report-ready conclusion tables.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os
import pandas as pd
import numpy as np
import glob

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

print("Project exists:", os.path.exists(project_path))
print("Metrics path exists:", os.path.exists(metrics_path))
print("Models path exists:", os.path.exists(models_path))

Project exists: True
Metrics path exists: True
Models path exists: True


In [3]:
metric_files = sorted(glob.glob(f"{metrics_path}/*.csv"))

print("Metric files:", len(metric_files))

for file in metric_files:
    print(os.path.basename(file))

Metric files: 96
article_aware_turkish_bge_4question_results.csv
article_aware_turkish_bge_4question_score.csv
article_aware_turkish_bge_rag_testset_generation_results.csv
article_aware_turkish_bge_rag_testset_score.csv
article_aware_turkish_bge_rag_testset_scored.csv
base_rag_generation_results.csv
base_rag_manual_score.csv
base_rag_testset_generation_results.csv
base_vs_finetuned_llm_rag_eval_summary_20.csv
base_vs_finetuned_llm_rag_generation_results_20.csv
base_vs_finetuned_llm_rag_manual_score_summary_20.csv
base_vs_finetuned_llm_rag_manual_scored_20.csv
base_vs_finetuned_llm_rag_manual_scoring_template_20.csv
baseline_retrieval_details.csv
baseline_retrieval_keyword_metrics.csv
controlled_gold_context_base_vs_finetuned_results_10.csv
controlled_starlar_base_generation_intermediate_20.csv
controlled_starlar_base_vs_finetuned_auto_summary_20.csv
controlled_starlar_base_vs_finetuned_eval_summary_20.csv
controlled_starlar_base_vs_finetuned_generation_results_20.csv
controlled_starlar

In [4]:
def safe_read_csv(path):
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f"Loaded: {os.path.basename(path)} | shape={df.shape}")
        return df
    else:
        print(f"Missing: {path}")
        return None


def show_file_status(file_name):
    path = f"{metrics_path}/{file_name}"
    print(file_name)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size KB:", round(os.path.getsize(path) / 1024, 2))
    print("-" * 80)

In [5]:
important_files = [
    # Old / best RAG pipeline
    "source_aware_article_aware_turkish_bge_rag_testset_score.csv",
    "source_aware_article_aware_turkish_bge_rag_testset_scored.csv",

    # Coverage / drift
    "data_coverage_analysis_summary.csv",
    "evaluation_drift_diagnosis_summary.csv",
    "final_pipeline_selection_summary.csv",

    # Starlar data prep and fine-tune
    "starlar_llm_sft_v2_data_preparation_summary.csv",
    "mistral_legal_qlora_starlar_v2_800steps_training_log.csv",
    "mistral_legal_qlora_starlar_v2_800steps_summary.csv",

    # Starlar evaluation
    "controlled_starlar_base_vs_finetuned_generation_results_20.csv",
    "controlled_starlar_base_vs_finetuned_auto_summary_20.csv",
    "controlled_starlar_base_vs_finetuned_manual_scored_20.csv",
    "controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv",
]

for file in important_files:
    show_file_status(file)

source_aware_article_aware_turkish_bge_rag_testset_score.csv
Exists: True
Size KB: 0.28
--------------------------------------------------------------------------------
source_aware_article_aware_turkish_bge_rag_testset_scored.csv
Exists: True
Size KB: 37.38
--------------------------------------------------------------------------------
data_coverage_analysis_summary.csv
Exists: True
Size KB: 0.68
--------------------------------------------------------------------------------
evaluation_drift_diagnosis_summary.csv
Exists: True
Size KB: 0.29
--------------------------------------------------------------------------------
final_pipeline_selection_summary.csv
Exists: True
Size KB: 0.69
--------------------------------------------------------------------------------
starlar_llm_sft_v2_data_preparation_summary.csv
Exists: True
Size KB: 0.53
--------------------------------------------------------------------------------
mistral_legal_qlora_starlar_v2_800steps_training_log.csv
Exists: True

In [6]:
old_best_score_path = f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_score.csv"
old_best_scored_path = f"{metrics_path}/source_aware_article_aware_turkish_bge_rag_testset_scored.csv"

old_best_score_df = safe_read_csv(old_best_score_path)
old_best_scored_df = safe_read_csv(old_best_scored_path)

if old_best_score_df is not None:
    display(old_best_score_df)

if old_best_scored_df is not None:
    print(old_best_scored_df.columns.tolist())
    display(old_best_scored_df.head())

Loaded: source_aware_article_aware_turkish_bge_rag_testset_score.csv | shape=(1, 10)
Loaded: source_aware_article_aware_turkish_bge_rag_testset_scored.csv | shape=(20, 16)


,method,manual_accuracy,valid_sample_count,total_sample_count,candidate_k,final_context_k,alpha,hybrid_weight,rerank_weight,article_bonus_weight
0,Source-Aware + Article-Aware Retrieval + Turki...,0.421053,19,20,10,3,0.5,0.7,0.3,0.25


['question', 'expected_answer', 'generated_answer', 'clean_generated_answer', 'top1_chunk_id', 'top1_source', 'top1_source_filter', 'top1_context', 'top1_article_bonus', 'top1_original_rank', 'top1_rerank_rank', 'top1_rerank_score', 'top1_fusion_score', 'retrieved_contexts', 'is_valid_sample', 'manual_score']


,question,expected_answer,generated_answer,clean_generated_answer,top1_chunk_id,top1_source,top1_source_filter,top1_context,top1_article_bonus,top1_original_rank,top1_rerank_rank,top1_rerank_score,top1_fusion_score,retrieved_contexts,is_valid_sample,manual_score
0,Anayasanın 101. Maddesiyle İlgili Tartışmalar ...,Cumhurbaşkanının seçilme şartlarının sınırları...,"Anayasanın 101. Maddesiyle ilgili tartışmalar,...","Anayasanın 101. Maddesiyle ilgili tartışmalar,...",chunk_000537,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,1.0,1,1,0.023523,1.00,Cumhurbaşkanlığı seçiminde birinci oylamada ge...,True,0.0
1,"Bir grup vatandaş, belirli bir etnik grubun di...","Anayasanın 10. Maddesi, herkesin kanun önünde ...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...","Anayasanın 10. Maddesi, herkesin dil, ırk, ren...",chunk_000274,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",1.0,1,1,0.027961,1.00,"Madde 10 – Herkes, dil, ırk, renk, cinsiyet, s...",True,0.5
2,"Bir grup akademisyen, yaşama hakkının sınırlan...","Evet, Anayasanın 17. Maddesi, herkesin yaşama ...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...","Hayır. Anayasanın 17. Maddesi, herkesin yaşama...",chunk_000373,Türkiye Cumhuriyeti Anayasası,Türkiye Cumhuriyeti Anayasası,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",1.0,1,1,0.385170,1.00,"Madde 17 – Herkes, yaşama, maddi ve manevi var...",True,0.5
3,Geçici madde 20 ne zaman eklendi?,20 mayıs 2016 tarihinde.,Geçici madde 20 13/5/1981 gün eklendi.,Geçici madde 20 13/5/1981 gün eklendi.,chunk_003745,Türkiye Cumhuriyeti İş Kanunu,NaN,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,1.0,1,1,0.438255,1.00,31/7/2008-5797/10 md.) Bu fıkrada düzenlenen t...,True,0.0
4,Videoda TCK 121 ihlali sabit değil mi?,Bu tür hususlar dosyalarında bilişimci bilirki...,"Evet, TCK 121 madde hakkının kullanılmasının e...","Evet, TCK 121 madde hakkının kullanılmasının e...",chunk_003431,Türk Ceza Kanunu,Türk Ceza Kanunu,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...,0.0,1,5,0.002067,0.76,ÜÇÜNCÜ KISIM\r\nTopluma Karşı Suçlar ALTINCI B...,True,0.5


In [7]:
if old_best_scored_df is not None:
    old_best_scored_df["is_valid_sample"] = old_best_scored_df["is_valid_sample"].astype(bool)

    old_valid_df = old_best_scored_df[old_best_scored_df["is_valid_sample"] == True].copy()

    old_best_full_accuracy = old_valid_df["manual_score"].mean()

    # Daha önce belirlediğimiz coverage issue indexleri
    coverage_issue_indices = [3, 11, 15]

    old_clean_df = old_best_scored_df[
        ~old_best_scored_df.index.isin(coverage_issue_indices)
    ].copy()

    old_clean_df = old_clean_df[old_clean_df["is_valid_sample"] == True].copy()

    old_best_coverage_clean_accuracy = old_clean_df["manual_score"].mean()

    print("Old best valid count:", len(old_valid_df))
    print("Old best full manual accuracy:", old_best_full_accuracy)
    print("Old best coverage-clean count:", len(old_clean_df))
    print("Old best coverage-clean accuracy:", old_best_coverage_clean_accuracy)
else:
    old_best_full_accuracy = 0.421053
    old_best_coverage_clean_accuracy = 0.46875

    print("Using fallback values:")
    print("Old best full:", old_best_full_accuracy)
    print("Old best coverage-clean:", old_best_coverage_clean_accuracy)

Old best valid count: 19
Old best full manual accuracy: 0.42105263157894735
Old best coverage-clean count: 16
Old best coverage-clean accuracy: 0.46875


In [8]:
rag_final_summary_df = pd.DataFrame([
    {
        "experiment_group": "Final RAG Pipeline",
        "method": "Source-aware + Article-aware + Turkish BGE reranker + improved legal prompt",
        "generator": "Base Mistral",
        "manual_accuracy": old_best_full_accuracy,
        "coverage_clean_accuracy": old_best_coverage_clean_accuracy,
        "selected_for_final": True,
        "notes": "Best completed end-to-end RAG pipeline. Coverage-clean score excludes known dataset/corpus coverage issues."
    }
])

display(rag_final_summary_df)

,experiment_group,method,generator,manual_accuracy,coverage_clean_accuracy,selected_for_final,notes
0,Final RAG Pipeline,Source-aware + Article-aware + Turkish BGE rer...,Base Mistral,0.421053,0.46875,True,Best completed end-to-end RAG pipeline. Covera...


In [9]:
drift_summary_path = f"{metrics_path}/evaluation_drift_diagnosis_summary.csv"
final_selection_path = f"{metrics_path}/final_pipeline_selection_summary.csv"

drift_summary_df = safe_read_csv(drift_summary_path)
final_selection_df = safe_read_csv(final_selection_path)

if drift_summary_df is not None:
    display(drift_summary_df)

if final_selection_df is not None:
    display(final_selection_df)

Loaded: evaluation_drift_diagnosis_summary.csv | shape=(1, 8)
Loaded: final_pipeline_selection_summary.csv | shape=(3, 8)


,same_question_count,total_questions,same_top1_chunk_count,same_top1_chunk_ratio,old_best_manual_accuracy,new_base_manual_accuracy,new_finetuned_manual_accuracy,diagnosis
0,20,20,13,0.65,0.421053,0.315789,0.263158,retrieval_and_prompt_generation_drift_confirmed


,method,retrieval_setup,generator,manual_accuracy,coverage_clean_accuracy,valid_sample_count,selected_for_final,notes
0,Old Best Final Pipeline,Source-aware + Article-aware + Turkish BGE rer...,Base Mistral with improved legal prompt,0.421053,0.46875,19,True,Best-performing final candidate. Used as final...
1,Notebook 14 Base Mistral RAG,Notebook 14 retrieval implementation,Base Mistral,0.315789,0.34375,19,False,Lower score due to retrieval and prompt/genera...
2,Notebook 14 Fine-tuned Mistral QLoRA RAG,Notebook 14 retrieval implementation,Fine-tuned Mistral QLoRA adapter,0.263158,0.28125,19,False,Fine-tuned LLM experiment; not selected for fi...


In [10]:
if drift_summary_df is not None:
    same_top1_ratio = drift_summary_df.loc[0, "same_top1_chunk_ratio"]
    same_top1_count = drift_summary_df.loc[0, "same_top1_chunk_count"]
else:
    same_top1_ratio = 0.65
    same_top1_count = 13

drift_interpretation_df = pd.DataFrame([{
    "analysis": "Evaluation drift check",
    "same_questions": "20 / 20",
    "same_top1_chunks": f"{same_top1_count} / 20",
    "same_top1_ratio": same_top1_ratio,
    "interpretation": "The later LLM evaluation did not exactly reproduce the previous best retrieval setup. Therefore, the lower first fine-tuned RAG result should not be treated as the final pipeline score."
}])

display(drift_interpretation_df)

,analysis,same_questions,same_top1_chunks,same_top1_ratio,interpretation
0,Evaluation drift check,20 / 20,13 / 20,0.65,The later LLM evaluation did not exactly repro...


In [11]:
starlar_ft_summary_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_800steps_summary.csv"
starlar_ft_log_path = f"{metrics_path}/mistral_legal_qlora_starlar_v2_800steps_training_log.csv"

starlar_ft_summary_df = safe_read_csv(starlar_ft_summary_path)
starlar_ft_log_df = safe_read_csv(starlar_ft_log_path)

if starlar_ft_summary_df is not None:
    display(starlar_ft_summary_df)

if starlar_ft_log_df is not None:
    display(starlar_ft_log_df.tail(20))

Loaded: mistral_legal_qlora_starlar_v2_800steps_summary.csv | shape=(1, 21)
Loaded: mistral_legal_qlora_starlar_v2_800steps_training_log.csv | shape=(97, 14)


,base_model,method,dataset,train_file,val_file,train_rows,val_rows,max_steps,save_steps,eval_steps,...,assistant_only_loss,learning_rate,gradient_accumulation_steps,per_device_train_batch_size,lora_r,lora_alpha,lora_dropout,adapter_output_path,best_model_checkpoint,best_metric
0,mistralai/Mistral-7B-Instruct-v0.2,QLoRA,Starlar LLM SFT v2,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,3200,400,800,50,50,...,True,0.0001,8,1,16,32,0.05,/content/drive/MyDrive/turkish_legal_rag/outpu...,/content/drive/MyDrive/turkish_legal_rag/outpu...,0.001149


,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
77,NaN,NaN,NaN,1.625,650,0.001186,230.1054,1.738,1.738,NaN,NaN,NaN,NaN,NaN
78,0.001597,0.005013,7.927377e-06,1.650,660,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
79,0.000135,0.007494,6.868399e-06,1.675,670,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
80,0.005241,0.005146,5.880104e-06,1.700,680,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
81,0.002206,0.000864,4.964111e-06,1.725,690,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
82,0.000118,0.000756,4.121921e-06,1.750,700,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,NaN,NaN,NaN,1.750,700,0.001150,229.3488,1.744,1.744,NaN,NaN,NaN,NaN,NaN
84,0.000115,0.000413,3.354915e-06,1.775,710,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,0.000538,0.001064,2.664349e-06,1.800,720,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,0.000387,0.000762,2.051355e-06,1.825,730,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
if starlar_ft_log_df is not None:
    eval_rows = starlar_ft_log_df[starlar_ft_log_df["eval_loss"].notna()].copy()

    first_eval_loss = eval_rows.iloc[0]["eval_loss"]
    final_eval_loss = eval_rows.iloc[-1]["eval_loss"]
    best_eval_loss = eval_rows["eval_loss"].min()
    best_eval_step = eval_rows.loc[eval_rows["eval_loss"].idxmin(), "step"]

    starlar_training_result_df = pd.DataFrame([{
        "model": "Mistral-7B-Instruct-v0.2",
        "fine_tune_method": "QLoRA",
        "dataset": "Starlar LLM SFT v2",
        "max_steps": 800,
        "assistant_only_loss": True,
        "first_eval_loss": first_eval_loss,
        "final_eval_loss": final_eval_loss,
        "best_eval_loss": best_eval_loss,
        "best_eval_step": best_eval_step,
        "interpretation": "Validation loss decreased and reached its best value near the end of training, indicating a stable fine-tuning run."
    }])
else:
    starlar_training_result_df = pd.DataFrame([{
        "model": "Mistral-7B-Instruct-v0.2",
        "fine_tune_method": "QLoRA",
        "dataset": "Starlar LLM SFT v2",
        "max_steps": 800,
        "assistant_only_loss": True,
        "first_eval_loss": 0.004296,
        "final_eval_loss": 0.001149,
        "best_eval_loss": 0.001149,
        "best_eval_step": 800,
        "interpretation": "Validation loss decreased strongly during training."
    }])

display(starlar_training_result_df)

,model,fine_tune_method,dataset,max_steps,assistant_only_loss,first_eval_loss,final_eval_loss,best_eval_loss,best_eval_step,interpretation
0,Mistral-7B-Instruct-v0.2,QLoRA,Starlar LLM SFT v2,800,True,0.004296,0.001149,0.001149,800,Validation loss decreased and reached its best...


In [13]:
controlled_auto_summary_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_auto_summary_20.csv"
controlled_manual_summary_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv"
controlled_results_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_generation_results_20.csv"
controlled_scored_path = f"{metrics_path}/controlled_starlar_base_vs_finetuned_manual_scored_20.csv"

controlled_auto_summary_df = safe_read_csv(controlled_auto_summary_path)
controlled_manual_summary_df = safe_read_csv(controlled_manual_summary_path)
controlled_results_df = safe_read_csv(controlled_results_path)
controlled_scored_df = safe_read_csv(controlled_scored_path)

if controlled_auto_summary_df is not None:
    display(controlled_auto_summary_df)

if controlled_manual_summary_df is not None:
    display(controlled_manual_summary_df)

Loaded: controlled_starlar_base_vs_finetuned_auto_summary_20.csv | shape=(2, 4)
Loaded: controlled_starlar_base_vs_finetuned_manual_score_summary_20.csv | shape=(2, 6)
Loaded: controlled_starlar_base_vs_finetuned_generation_results_20.csv | shape=(20, 17)
Loaded: controlled_starlar_base_vs_finetuned_manual_scored_20.csv | shape=(20, 22)


,method,mean_token_f1,mean_text_similarity,source_citation_rate
0,Base Mistral,0.184815,0.155437,0.25
1,Starlar Fine-tuned Mistral QLoRA,0.965736,0.969455,0.85


,method,manual_accuracy,mean_token_f1,mean_text_similarity,source_citation_rate,sample_count
0,Base Mistral,0.400,0.184815,0.155437,0.25,20
1,Starlar Fine-tuned Mistral QLoRA,0.925,0.965736,0.969455,0.85,20


In [14]:
if controlled_manual_summary_df is not None:
    base_row = controlled_manual_summary_df[
        controlled_manual_summary_df["method"].str.contains("Base", case=False, na=False)
    ].iloc[0]

    ft_row = controlled_manual_summary_df[
        controlled_manual_summary_df["method"].str.contains("Fine", case=False, na=False)
    ].iloc[0]

    base_manual = base_row["manual_accuracy"]
    ft_manual = ft_row["manual_accuracy"]

    base_f1 = base_row["mean_token_f1"]
    ft_f1 = ft_row["mean_token_f1"]

    base_sim = base_row["mean_text_similarity"]
    ft_sim = ft_row["mean_text_similarity"]

    base_citation = base_row["source_citation_rate"]
    ft_citation = ft_row["source_citation_rate"]
else:
    base_manual = 0.400
    ft_manual = 0.925

    base_f1 = 0.184815
    ft_f1 = 0.965736

    base_sim = 0.155437
    ft_sim = 0.969455

    base_citation = 0.25
    ft_citation = 0.85

controlled_improvement_df = pd.DataFrame([
    {
        "metric": "Manual accuracy",
        "base_mistral": base_manual,
        "starlar_finetuned_mistral": ft_manual,
        "absolute_improvement": ft_manual - base_manual
    },
    {
        "metric": "Mean token F1",
        "base_mistral": base_f1,
        "starlar_finetuned_mistral": ft_f1,
        "absolute_improvement": ft_f1 - base_f1
    },
    {
        "metric": "Mean text similarity",
        "base_mistral": base_sim,
        "starlar_finetuned_mistral": ft_sim,
        "absolute_improvement": ft_sim - base_sim
    },
    {
        "metric": "Source citation rate",
        "base_mistral": base_citation,
        "starlar_finetuned_mistral": ft_citation,
        "absolute_improvement": ft_citation - base_citation
    }
])

display(controlled_improvement_df)

,metric,base_mistral,starlar_finetuned_mistral,absolute_improvement
0,Manual accuracy,0.400000,0.925000,0.525000
1,Mean token F1,0.184815,0.965736,0.780921
2,Mean text similarity,0.155437,0.969455,0.814018
3,Source citation rate,0.250000,0.850000,0.600000


In [15]:
final_system_decision_df = pd.DataFrame([
    {
        "component": "Retrieval",
        "selected_method": "Source-aware + article-aware retrieval",
        "reason": "Improved legal source selection and article-number matching."
    },
    {
        "component": "Reranking",
        "selected_method": "Turkish BGE reranker",
        "reason": "Improved ranking quality over baseline retrieval."
    },
    {
        "component": "Prompting",
        "selected_method": "Improved legal prompt",
        "reason": "Reduced unsupported generation and encouraged concise legal answers."
    },
    {
        "component": "End-to-end RAG generator",
        "selected_method": "Base Mistral for original 20-question RAG benchmark",
        "reason": "Best completed end-to-end RAG pipeline result was measured with this setup."
    },
    {
        "component": "Fine-tuned generator experiment",
        "selected_method": "Starlar fine-tuned Mistral QLoRA",
        "reason": "Clearly improved controlled source-grounded answer generation compared to base Mistral."
    }
])

display(final_system_decision_df)

,component,selected_method,reason
0,Retrieval,Source-aware + article-aware retrieval,Improved legal source selection and article-nu...
1,Reranking,Turkish BGE reranker,Improved ranking quality over baseline retrieval.
2,Prompting,Improved legal prompt,Reduced unsupported generation and encouraged ...
3,End-to-end RAG generator,Base Mistral for original 20-question RAG benc...,Best completed end-to-end RAG pipeline result ...
4,Fine-tuned generator experiment,Starlar fine-tuned Mistral QLoRA,Clearly improved controlled source-grounded an...


In [16]:
final_master_results_df = pd.DataFrame([
    {
        "category": "End-to-end RAG",
        "experiment": "Best RAG pipeline",
        "setup": "Source-aware + article-aware retrieval + Turkish BGE reranker + improved prompt + Base Mistral",
        "main_metric": "Manual accuracy",
        "base_or_previous": None,
        "final_or_finetuned": old_best_full_accuracy,
        "coverage_clean": old_best_coverage_clean_accuracy,
        "interpretation": "Best completed RAG pipeline on the original 20-question evaluation set."
    },
    {
        "category": "LLM fine-tuning",
        "experiment": "Controlled Starlar gold-context evaluation",
        "setup": "Base Mistral vs Starlar fine-tuned Mistral QLoRA",
        "main_metric": "Manual accuracy",
        "base_or_previous": base_manual,
        "final_or_finetuned": ft_manual,
        "coverage_clean": None,
        "interpretation": "Fine-tuning substantially improved source-grounded answer generation when the correct context was provided."
    },
    {
        "category": "LLM fine-tuning",
        "experiment": "Controlled Starlar gold-context evaluation",
        "setup": "Base Mistral vs Starlar fine-tuned Mistral QLoRA",
        "main_metric": "Mean token F1",
        "base_or_previous": base_f1,
        "final_or_finetuned": ft_f1,
        "coverage_clean": None,
        "interpretation": "Automatic token-level overlap strongly improved after fine-tuning."
    },
    {
        "category": "LLM fine-tuning",
        "experiment": "Controlled Starlar gold-context evaluation",
        "setup": "Base Mistral vs Starlar fine-tuned Mistral QLoRA",
        "main_metric": "Source citation rate",
        "base_or_previous": base_citation,
        "final_or_finetuned": ft_citation,
        "coverage_clean": None,
        "interpretation": "Fine-tuned model learned the expected source/citation answer format better."
    }
])

display(final_master_results_df)

,category,experiment,setup,main_metric,base_or_previous,final_or_finetuned,coverage_clean,interpretation
0,End-to-end RAG,Best RAG pipeline,Source-aware + article-aware retrieval + Turki...,Manual accuracy,NaN,0.421053,0.46875,Best completed RAG pipeline on the original 20...
1,LLM fine-tuning,Controlled Starlar gold-context evaluation,Base Mistral vs Starlar fine-tuned Mistral QLoRA,Manual accuracy,0.400000,0.925000,NaN,Fine-tuning substantially improved source-grou...
2,LLM fine-tuning,Controlled Starlar gold-context evaluation,Base Mistral vs Starlar fine-tuned Mistral QLoRA,Mean token F1,0.184815,0.965736,NaN,Automatic token-level overlap strongly improve...
3,LLM fine-tuning,Controlled Starlar gold-context evaluation,Base Mistral vs Starlar fine-tuned Mistral QLoRA,Source citation rate,0.250000,0.850000,NaN,Fine-tuned model learned the expected source/c...


In [17]:
final_report_paragraphs = {
    "rag_result": f"""
The best completed end-to-end RAG pipeline used source-aware retrieval, article-aware retrieval, a Turkish BGE reranker, and an improved legal prompt.
On the original 20-question manually evaluated test set, this configuration achieved a manual accuracy of {old_best_full_accuracy:.4f}.
After excluding known coverage/problematic samples, the coverage-clean accuracy increased to {old_best_coverage_clean_accuracy:.4f}.
""".strip(),

    "fine_tuning_result": f"""
The Starlar LLM SFT v2 fine-tuning experiment showed a clear improvement in controlled source-grounded generation.
In the controlled Starlar gold-context evaluation, the base Mistral model achieved a manual accuracy of {base_manual:.3f}, while the Starlar fine-tuned Mistral QLoRA model achieved {ft_manual:.3f}.
Automatic metrics showed the same trend: mean token F1 increased from {base_f1:.4f} to {ft_f1:.4f}, mean text similarity increased from {base_sim:.4f} to {ft_sim:.4f}, and source citation rate increased from {base_citation:.2f} to {ft_citation:.2f}.
""".strip(),

    "final_interpretation": """
These results indicate that retrieval-side improvements were important for the end-to-end RAG pipeline, while LLM fine-tuning substantially improved the model's ability to produce source-grounded Turkish legal answers when the correct context was provided.
Therefore, the final project reports both the best RAG pipeline performance and the controlled LLM fine-tuning improvement.
""".strip()
}

for title, paragraph in final_report_paragraphs.items():
    print("=" * 100)
    print(title)
    print(paragraph)

rag_result
The best completed end-to-end RAG pipeline used source-aware retrieval, article-aware retrieval, a Turkish BGE reranker, and an improved legal prompt. 
On the original 20-question manually evaluated test set, this configuration achieved a manual accuracy of 0.4211. 
After excluding known coverage/problematic samples, the coverage-clean accuracy increased to 0.4688.
fine_tuning_result
The Starlar LLM SFT v2 fine-tuning experiment showed a clear improvement in controlled source-grounded generation. 
In the controlled Starlar gold-context evaluation, the base Mistral model achieved a manual accuracy of 0.400, while the Starlar fine-tuned Mistral QLoRA model achieved 0.925. 
Automatic metrics showed the same trend: mean token F1 increased from 0.1848 to 0.9657, mean text similarity increased from 0.1554 to 0.9695, and source citation rate increased from 0.25 to 0.85.
final_interpretation
These results indicate that retrieval-side improvements were important for the end-to-end RA

In [18]:
final_results_summary_path = f"{metrics_path}/final_results_master_summary.csv"
final_system_decision_path = f"{metrics_path}/final_system_decision.csv"
controlled_improvement_path = f"{metrics_path}/final_controlled_finetuning_improvement_summary.csv"
report_paragraphs_path = f"{metrics_path}/final_report_paragraphs.csv"

final_master_results_df.to_csv(
    final_results_summary_path,
    index=False,
    encoding="utf-8-sig"
)

final_system_decision_df.to_csv(
    final_system_decision_path,
    index=False,
    encoding="utf-8-sig"
)

controlled_improvement_df.to_csv(
    controlled_improvement_path,
    index=False,
    encoding="utf-8-sig"
)

pd.DataFrame([
    {"section": key, "paragraph": value}
    for key, value in final_report_paragraphs.items()
]).to_csv(
    report_paragraphs_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(final_results_summary_path)
print(final_system_decision_path)
print(controlled_improvement_path)
print(report_paragraphs_path)

Saved:
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_results_master_summary.csv
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_system_decision.csv
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_controlled_finetuning_improvement_summary.csv
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_report_paragraphs.csv


In [19]:
final_files = [
    final_results_summary_path,
    final_system_decision_path,
    controlled_improvement_path,
    report_paragraphs_path
]

print("FINAL CHECK")
print("=" * 80)

for file in final_files:
    print(file)
    print("Exists:", os.path.exists(file))
    if os.path.exists(file):
        print("Size KB:", round(os.path.getsize(file) / 1024, 2))
    print("-" * 80)

print("Notebook 21 completed.")

FINAL CHECK
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_results_master_summary.csv
Exists: True
Size KB: 1.01
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_system_decision.csv
Exists: True
Size KB: 0.62
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_controlled_finetuning_improvement_summary.csv
Exists: True
Size KB: 0.28
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/final_report_paragraphs.csv
Exists: True
Size KB: 1.29
--------------------------------------------------------------------------------
Notebook 21 completed.
